# Task 2: QLoRA Fine-Tuning Qwen2.5-Coder-7B-Instruct
- **Dữ liệu**: 477 mẫu synthetic cân bằng (Claude gán nhãn theo taxonomy mới) + 32 bài Gold của thầy (nhân đôi) = 541 mẫu ChatML.
- **Phương pháp**: QLoRA 4-bit SFT với Completion-only Loss Masking.
- **Đánh giá**: Đo lường Macro-F1 trực tiếp trên 32 bài Gold trước khi nộp.
- **Đầu ra**: `/kaggle/working/task2_predictions.json` chuẩn format nộp bài.

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.48,<5" bitsandbytes>=0.42.0 peft>=0.9.0 accelerate>=0.27.0

In [ ]:
import os, sys, shutil
from pathlib import Path

# Clone hoặc cập nhật repo mới nhất từ branch task-2-split-run
repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    !git clone -b task-2-split-run https://github.com/tiendinhquang2104/llm-final-project.git /kaggle/working/repo
else:
    !git -C /kaggle/working/repo fetch origin task-2-split-run
    !git -C /kaggle/working/repo reset --hard origin/task-2-split-run

os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo")
print("Working directory:", os.getcwd())

In [ ]:
# Step 1: Chuẩn bị dữ liệu huấn luyện (tập hợp 541 mẫu đã sinh sẵn trong repo)
!python scripts/prepare_task2_sft.py

In [ ]:
# Step 2: Huấn luyện QLoRA adapter trên Qwen2.5-Coder-7B-Instruct (khoảng 25-35 phút trên T4)
!python scripts/train_task2_sft.py \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --data_path data/processed/task2_sft_train.jsonl \
    --output_dir /kaggle/working/qlora_adapter \
    --epochs 1 \
    --batch_size 1 \
    --grad_accum 4 \
    --max_seq_len 1536 \
    --lr 2e-4

# Kiểm tra chắc chắn adapter đã được lưu thành công
adapter_ready = os.path.exists('/kaggle/working/qlora_adapter/adapter_config.json') or any(Path('/kaggle/working/qlora_adapter').glob('checkpoint-*/adapter_config.json'))
assert adapter_ready, 'Lỗi: Không tìm thấy adapter_config.json sau khi train!'
print('==> Huấn luyện thành công, adapter đã sẵn sàng!')

In [ ]:
# Step 3A: Đánh giá cục bộ trên 32 bài Gold của thầy (tính Macro-F1, Micro-F1 chi tiết)
!python scripts/infer_task2_sft.py \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_dir /kaggle/working/qlora_adapter \
    --data_dir data/sample_dataset \
    --output_file /kaggle/working/gold_eval_predictions.json

In [ ]:
# Step 3B: Chạy suy luận trên tập kiểm thử Public Test (328 mẫu nộp bài)
!python scripts/infer_task2_sft.py \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_dir /kaggle/working/qlora_adapter \
    --data_dir data/public_test_release \
    --output_file /kaggle/working/task2_predictions.json

print('Hoàn thành! File nộp bài đã lưu tại /kaggle/working/task2_predictions.json')

In [ ]:
# Step 4: Kiểm tra định dạng file nộp bài và phân bố nhãn
import json
from collections import Counter

with open('/kaggle/working/task2_predictions.json', encoding='utf-8') as f:
    preds = json.load(f)

print(f'Tổng số mẫu dự đoán: {len(preds)}')
assert len(preds) == 328, f'Kỳ vọng 328 mẫu, nhưng tìm thấy {len(preds)}'

all_labels = [l for p in preds for l in p['output']['taxonomy_error']]
counts = Counter(all_labels)
print(f'\nPhân bố các nhãn taxonomy (tổng số nhãn: {len(all_labels)}):')
for label, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  - {label}: {count}')
